# HDT 1: Pandas, SQL y DuckDB

**Ciencia de Datos, Sección A** · Asignada: martes 28 de julio · **Entrega: martes 4 de agosto, 23:59**

**Nombre:** _Victor Saravia_ 
**Carné:** _20240060_

Completar las celdas marcadas con `# ¿Qué va aquí?`. Cada ejercicio incluye una verificación comentada: descomentar para comprobar el resultado. Antes de entregar: **Kernel → Restart & Run All** (un notebook que no corre de arriba a abajo pierde 0.5 pts).

AI: resolver sin AI. Si se usó para entender un concepto, anotarlo en la mini-bitácora del final.

## Setup

Si falta DuckDB: descomentar la línea de instalación, ejecutar la celda una vez y volver a comentarla.

In [1]:
# !uv add duckdb    (en terminal)  o descomentar:  %pip install duckdb
import pandas as pd
import duckdb

URL = ("https://raw.githubusercontent.com/"
       "mwaskom/seaborn-data/master/penguins.csv")
penguins = pd.read_csv(URL)

# Tabla de nombres científicos (para los JOIN)
especies = pd.DataFrame({
    "species": ["Adelie", "Chinstrap", "Gentoo"],
    "nombre_cientifico": ["Pygoscelis adeliae",
                          "Pygoscelis antarcticus",
                          "Pygoscelis papua"],
})
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


## Parte A · Pandas (1.0 pt)

### Ejercicio 1 (0.10): cargar y explorar

Mostrar: (a) el `shape` del DataFrame, (b) los `dtypes`, y (c) cuántos nulos tiene **cada columna**.

In [7]:
# ¿Qué va aquí? (tres expresiones, una por inciso)
print(penguins.shape)
# Verificación: el dataset original tiene 344 filas y 7 columnas,
print(penguins.dtypes)
# y la columna sex es la que más nulos tiene (11).
print(f"{penguins.sex.isna().sum()} serían nulos") # Para este caso usamos "isna()" porque no son números lo que se quiere analizar, por lo que usamos True/False no count al querer los nulos.

# na = null

(344, 7)
species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object
11 serían nulos


### Ejercicio 2 (0.15): limpieza mínima

Crear un DataFrame `limpio` **sin** las filas que tengan nulo en cualquier columna. Reportar con un `print` cuántas filas se perdieron respecto al original.

In [8]:
limpio = penguins.dropna()   # ¿Qué va aquí?

print(f"Se perdieron {penguins.shape[0] - limpio.shape[0]} filas")

# Verificación (descomentar):
assert limpio.shape[0] == 333 and limpio.isna().sum().sum() == 0

Se perdieron 11 filas


### Ejercicio 3 (0.15): máscaras y orden

De `limpio`: los pingüinos de la isla **Biscoe** con masa corporal **mayor a 4500 g**, ordenados de mayor a menor masa. Mostrar solo las columnas `species`, `island`, `body_mass_g`.

In [17]:
pesados_biscoe = limpio[(limpio["island"] == "Biscoe") & (limpio["body_mass_g"] > 4500)][
["species", "island", "body_mass_g"]].sort_values("body_mass_g", ascending=False)   # ¿Qué va aquí?
print(pesados_biscoe)

# Comentario 1: Me tocó investigar el sort_values, pues no recordaba que función ordenaba en forma descendente.
# Comentario 2: Me sirvió bastante el ejemplo lo de los tips en el notebook, en explorar y seleccionar, de clase para poder hacer el filtrado de los datos.

# Verificación (descomentar):
assert (pesados_biscoe["island"] == "Biscoe").all()
assert (pesados_biscoe["body_mass_g"] > 4500).all()
assert pesados_biscoe["body_mass_g"].is_monotonic_decreasing

    species  island  body_mass_g
237  Gentoo  Biscoe       6300.0
253  Gentoo  Biscoe       6050.0
337  Gentoo  Biscoe       6000.0
297  Gentoo  Biscoe       6000.0
331  Gentoo  Biscoe       5950.0
..      ...     ...          ...
306  Gentoo  Biscoe       4600.0
111  Adelie  Biscoe       4600.0
248  Gentoo  Biscoe       4600.0
328  Gentoo  Biscoe       4575.0
225  Gentoo  Biscoe       4550.0

[106 rows x 3 columns]


### Ejercicio 4 (0.20): groupby con dos funciones

Masa corporal por **especie y sexo**: el **promedio** y el **conteo**, en una sola operación con `groupby` + `agg`.

In [ ]:
resumen = ...   # ¿Qué va aquí?
resumen

# Verificación: el grupo más pesado debe ser Gentoo macho (~5485 g en promedio).

### Ejercicio 5 (0.20): columna derivada

Agregar a `limpio` una columna `bill_ratio` = largo del pico / profundidad del pico. Mostrar el promedio de `bill_ratio` **por especie**, ordenado descendente. ¿Qué especie tiene el pico proporcionalmente más alargado?

In [ ]:
# ¿Qué va aquí?


# Verificación: Gentoo debe quedar de primero (~3.2).

### Ejercicio 6 (0.20): merge

Unir `limpio` con la tabla `especies` para que cada fila tenga su `nombre_cientifico`. Mostrar una fila de cada especie para comprobar.

In [ ]:
con_nombres = ...   # ¿Qué va aquí?

# con_nombres.drop_duplicates("species")[["species", "nombre_cientifico"]]

# Verificación (descomentar):
# assert con_nombres.shape[0] == limpio.shape[0]
# assert "nombre_cientifico" in con_nombres.columns

## Parte B · SQL con DuckDB (0.8 pt)

DuckDB consulta directamente los DataFrames en memoria: `duckdb.sql("SELECT ... FROM limpio")`. Cerrar cada consulta con `.df()` para ver el resultado como DataFrame.

### Ejercicio 7 (0.20): SELECT / WHERE / ORDER BY

El ejercicio 3, ahora en SQL: especie, isla y masa de los pingüinos de Biscoe con masa mayor a 4500 g, ordenados de mayor a menor.

In [ ]:
q7 = """
-- ¿Qué va aquí?
"""
duckdb.sql(q7).df()

# Verificación: debe dar las mismas filas que el ejercicio 3.

### Ejercicio 8 (0.20): GROUP BY + HAVING

Especies cuya masa corporal **promedio** supera los 4000 g, con su promedio redondeado.

In [ ]:
q8 = """
-- ¿Qué va aquí?
"""
duckdb.sql(q8).df()

# Verificación: solo una especie debe aparecer. ¿Cuál? Comparar con el resultado
# del ejercicio 4.

### Ejercicio 9 (0.20): JOIN

El ejercicio 6, ahora en SQL: unir `limpio` con `especies` y mostrar especie, nombre científico y masa promedio por especie.

In [ ]:
q9 = """
-- ¿Qué va aquí?
"""
duckdb.sql(q9).df()

# Verificación: 3 filas, una por especie, cada una con su Pygoscelis.

### Ejercicio 10 (0.20): window function

Los **3 pingüinos más pesados de cada especie**, usando `RANK() OVER (PARTITION BY ... ORDER BY ...)`. Pista de la sesión 3: la window function se calcula en una subconsulta y se filtra afuera.

In [ ]:
q10 = """
-- ¿Qué va aquí?
"""
duckdb.sql(q10).df()

# Verificación: alrededor de 9 filas (3 por especie; puede haber empates),
# y el rango nunca debe pasar de 3.

## Parte C · Criterio (0.2 pt)

### Ejercicio 11 (0.20)

Los mismos análisis se resolvieron en Pandas y en SQL. En 3-4 líneas, **con base en el trabajo de esta hoja** (no de memoria): ¿cuándo conviene cada herramienta? Mencionar al menos una operación que resultó más natural en cada una.

_(Responder editando esta celda)_

**Respuesta:** ...

## Mini-bitácora de AI (opcional, no penaliza)

Si se usó AI para entender algún concepto, anotar aquí qué se preguntó y qué se entendió. Si no se usó, escribir "No se usó".

- ...

## Anexo: repaso de las sesiones 2 y 3

Regla de los ejercicios de NumPy: **sin ciclos `for`**.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

### A1 (sesión 2): z-score sin loops

Normalizar un array: restar la media y dividir entre la desviación estándar.

In [ ]:
alturas = rng.normal(170, 10, size=1000)

def z_score(x):
    # ¿Qué va aquí? (sin for)
    pass

z = z_score(alturas)
# Verificación (descomentar):
# print(round(z.mean(), 4), round(z.std(), 4))  # ~0 y ~1

### A2 (sesión 2): distancias con broadcasting

Distancia euclidiana de cada punto a un centro, sin loops.

In [ ]:
puntos = rng.normal(size=(500, 2))   # 500 puntos en 2D
centro = np.array([1.0, 1.0])

# ¿Qué va aquí?
# Pista: (puntos - centro) usa broadcasting (500,2) - (2,)
# Luego: elevar al cuadrado, sumar con axis=1, sacar raíz
distancias = ...

# ¿Cuántos puntos están a menos de 1 del centro?
cercanos = ...

# Verificación (descomentar):
# print(distancias.shape)  # (500,)
# print(cercanos)

### A3 (sesión 3): propinas por día y turno

Dataset `tips` (propinas de un restaurante). `pct` = propina como fracción de la cuenta.

In [ ]:
URL_TIPS = ("https://raw.githubusercontent.com/"
            "mwaskom/seaborn-data/master/tips.csv")
tips = pd.read_csv(URL_TIPS)
tips["pct"] = tips["tip"] / tips["total_bill"]
tips.head()

In [ ]:
# a) porcentaje medio de propina por dia
# b) por dia Y turno (day, time), en una tabla
# c) el dia con el mayor porcentaje medio
# d) numero de mesas por dia

resumen = ...  # ¿Que va aqui? (usar agg)

# Verificacion: resumen debe tener 4 filas
# print(resumen.shape)